# Dia 4 — Databricks: Delta Lake + MLflow
### Curso: Introduccion a Herramientas de Computo en la Nube · Azure

---

Este notebook continua donde termino el del Dia 3.  
Asegurate de haber ejecutado el notebook `dia3_databricks.ipynb` antes de empezar —  
necesitamos que `df_spark` y las variables de credenciales esten en memoria.

| Seccion | Contenido |
|---------|----------|
| 1 | Reconfigurar credenciales y reconectar Blob |
| 2 | Delta Lake: crear tabla, consultar, time travel |
| 3 | Delta Lake: MERGE (upsert) |
| 4 | MLflow: tracking de experimentos en Databricks |
| 5 | MLflow: entrenar y comparar modelos |
| 6 | MLflow: registrar el mejor modelo |

> **Cluster:** usa el mismo cluster Single Node del Dia 3. Delta Lake y MLflow vienen preinstalados en Databricks Runtime 13.x+.

---
## Seccion 1 — Reconfigurar credenciales

Si el cluster fue reiniciado, volvemos a configurar las variables y reconectamos Blob.

In [ ]:
# Credenciales — mismas que el Dia 3
STORAGE_ACCOUNT_NAME = "TU_CUENTA"
STORAGE_ACCOUNT_KEY  = "TU_CLAVE"
BLOB_CONTAINER       = "mi-contenedor"
BLOB_ARCHIVO_CSV     = "datos.csv"

PG_HOST     = "TU-SERVIDOR.postgres.database.azure.com"
PG_PORT     = "5432"
PG_DATABASE = "postgres"
PG_USER     = "TU_USUARIO"
PG_PASSWORD = "TU_CONTRASENA"
JDBC_URL    = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}?sslmode=require"

jdbc_props = {
    "user"    : PG_USER,
    "password": PG_PASSWORD,
    "driver"  : "org.postgresql.Driver",
    "sslmode" : "require"
}

# Reconfigurar ABFS
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

abfs_path = f"abfss://{BLOB_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/{BLOB_ARCHIVO_CSV}"

# Releer el CSV desde Blob
df_spark = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(abfs_path)
)

print(f"Listo. {df_spark.count()} filas cargadas desde Blob.")

---
## Seccion 2 — Delta Lake: crear tabla y time travel

Delta Lake es una capa de almacenamiento transaccional sobre Parquet.  
Agrega un **transaction log** que habilita ACID transactions y versionado.

**Flujo:**
1. Escribir el DataFrame como Delta Table en Blob Storage
2. Consultar la tabla
3. Hacer un update y ver el historial de versiones
4. Consultar una version anterior (time travel)

In [ ]:
# Ruta en Blob donde guardaremos la Delta Table
delta_path = f"abfss://{BLOB_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/delta/registros"

# Escribir el DataFrame como Delta Table
# mode='overwrite' crea la tabla o la reemplaza si ya existe
(
    df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .save(delta_path)
)

print(f"Delta Table creada en: {delta_path}")
print(f"Filas escritas: {df_spark.count()}")

In [ ]:
# Leer la Delta Table (igual que cualquier DataFrame de Spark)
df_delta = spark.read.format("delta").load(delta_path)

print(f"Filas en la Delta Table: {df_delta.count()}")
display(df_delta)

In [ ]:
# Registrar como vista SQL para usar Spark SQL
df_delta.createOrReplaceTempView("registros_delta")

# Consulta SQL sobre la Delta Table
resultado = spark.sql("""
    SELECT
        categoria,
        COUNT(*)                    AS total,
        ROUND(AVG(valor), 2)        AS promedio_valor,
        ROUND(MAX(valor), 2)        AS max_valor
    FROM registros_delta
    GROUP BY categoria
    ORDER BY promedio_valor DESC
""")
display(resultado)

In [ ]:
from delta.tables import DeltaTable

# Obtener referencia a la Delta Table
dt = DeltaTable.forPath(spark, delta_path)

# Ver el historial de versiones
print("Historial de la Delta Table:")
display(dt.history())

In [ ]:
# Hacer un update: incrementar el valor de todos los registros de 'Ventas' en 10%
dt.update(
    condition = "categoria = 'Ventas'",
    set       = {"valor": "valor * 1.10"}
)

print("Update aplicado: valor de Ventas incrementado 10%.")

# Ver el historial ahora — debe mostrar la version 1
display(dt.history().select("version", "timestamp", "operation", "operationParameters"))

In [ ]:
# Time travel: leer la version 0 (antes del update)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
df_v1 = spark.read.format("delta").option("versionAsOf", 1).load(delta_path)

# Comparar el promedio de Ventas entre versiones
from pyspark.sql import functions as F

avg_v0 = df_v0.filter("categoria = 'Ventas'").agg(F.round(F.avg("valor"), 2).alias("avg")).collect()[0]["avg"]
avg_v1 = df_v1.filter("categoria = 'Ventas'").agg(F.round(F.avg("valor"), 2).alias("avg")).collect()[0]["avg"]

print("Time travel — comparacion de versiones:")
print(f"  Version 0 (original)  avg Ventas: {avg_v0}")
print(f"  Version 1 (con update) avg Ventas: {avg_v1}")
print(f"  Diferencia: +{round(avg_v1 - avg_v0, 2)}")

---
## Seccion 3 — Delta Lake: MERGE (upsert)

MERGE es una de las operaciones mas utiles de Delta Lake.  
Permite insertar filas nuevas y actualizar las existentes en una sola operacion,
sin duplicados y sin reescribir toda la tabla.

In [ ]:
from pyspark.sql import functions as F

# Simular un batch de datos nuevos/actualizados
# Algunos nombres ya existen en la tabla, otros son nuevos
nuevos_datos = [
    ("2024-12-01", "Ana Torres",     9999.99, "Ventas"),       # nombre existente, valor nuevo
    ("2024-12-01", "Nuevo Usuario",   500.00, "Tecnologia"),   # nombre nuevo
    ("2024-12-02", "Otro Usuario",    750.50, "Finanzas"),     # nombre nuevo
]

df_nuevos = spark.createDataFrame(nuevos_datos, ["fecha", "nombre", "valor", "categoria"])

# MERGE: si el nombre ya existe -> actualizar valor; si no -> insertar
(
    dt.alias("tabla")
    .merge(
        df_nuevos.alias("nuevos"),
        "tabla.nombre = nuevos.nombre"
    )
    .whenMatchedUpdate(set={
        "valor"    : "nuevos.valor",
        "fecha"    : "nuevos.fecha",
        "categoria": "nuevos.categoria"
    })
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE ejecutado.")
print(f"Filas en la tabla ahora: {spark.read.format('delta').load(delta_path).count()}")

In [ ]:
# Verificar el resultado del MERGE
df_post_merge = spark.read.format("delta").load(delta_path)

# Ver las filas afectadas
display(
    df_post_merge.filter(
        F.col("nombre").isin("Ana Torres", "Nuevo Usuario", "Otro Usuario")
    )
)

# Historial completo
print("\nHistorial de versiones:")
display(dt.history().select("version", "timestamp", "operation"))

---
## Seccion 4 — MLflow: tracking de experimentos

MLflow viene preinstalado en Databricks. Cada vez que entrenas un modelo,
puedes registrar sus parametros y metricas en un **experiment**.

La UI de MLflow en Databricks permite comparar runs visualmente.
Para verla: barra izquierda > **Experiments**.

**Flujo de esta seccion:**
1. Preparar los datos para ML (desde la Delta Table)
2. Entrenar varios modelos con distintos hiperparametros
3. Loggear cada run con MLflow
4. Comparar los runs en la UI

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.ensemble         import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model     import Ridge
from sklearn.preprocessing    import LabelEncoder
from sklearn.model_selection  import train_test_split
from sklearn.metrics          import mean_absolute_error, r2_score, mean_squared_error

# Definir el nombre del experimento
# Databricks lo crea automaticamente si no existe
EXPERIMENT_NAME = "/Users/tu_usuario/curso_azure_dia4"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Experimento MLflow: {EXPERIMENT_NAME}")
print("Puedes verlo en: barra izquierda > Experiments")

In [ ]:
# Preparar datos para ML
# Convertimos la Delta Table a pandas para scikit-learn
df_ml = spark.read.format("delta").load(delta_path).toPandas()

# Encodificar la columna 'categoria' (texto -> numerico)
le = LabelEncoder()
df_ml["categoria_enc"] = le.fit_transform(df_ml["categoria"])

# Features y target
# Objetivo: predecir 'valor' a partir de 'categoria'
X = df_ml[["categoria_enc"]]
y = df_ml["valor"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset preparado.")
print(f"  Train : {len(X_train)} filas")
print(f"  Test  : {len(X_test)} filas")
print(f"  Clases: {list(le.classes_)}")

---
## Seccion 5 — MLflow: entrenar y comparar modelos

Entrenamos tres modelos distintos. Cada uno es un **run** dentro del mismo experimento.  
MLflow registra automaticamente los parametros, metricas y el modelo serializado.

In [ ]:
def entrenar_y_loggear(modelo, nombre, params, X_train, X_test, y_train, y_test):
    """
    Entrena un modelo, calcula metricas y logea todo en MLflow.
    Retorna el run_id para referencia posterior.
    """
    with mlflow.start_run(run_name=nombre) as run:

        # Loggear parametros del modelo
        mlflow.log_params(params)

        # Entrenar
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        # Calcular metricas
        mae  = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2   = r2_score(y_test, y_pred)

        # Loggear metricas
        mlflow.log_metric("mae",  mae)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2",   r2)

        # Loggear el modelo
        mlflow.sklearn.log_model(modelo, artifact_path="model")

        print(f"  Run: {nombre}")
        print(f"    MAE  : {mae:.2f}")
        print(f"    RMSE : {rmse:.2f}")
        print(f"    R2   : {r2:.4f}")
        print(f"    Run ID: {run.info.run_id}")
        print()

        return run.info.run_id


print("Entrenando modelos y loggeando en MLflow...\n")

run_ids = {}

# Modelo 1: Ridge Regression
run_ids["ridge"] = entrenar_y_loggear(
    Ridge(alpha=1.0), "Ridge_alpha1",
    {"model_type": "Ridge", "alpha": 1.0},
    X_train, X_test, y_train, y_test
)

# Modelo 2: Random Forest
run_ids["rf"] = entrenar_y_loggear(
    RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42),
    "RandomForest_50trees",
    {"model_type": "RandomForest", "n_estimators": 50, "max_depth": 5},
    X_train, X_test, y_train, y_test
)

# Modelo 3: Gradient Boosting
run_ids["gbm"] = entrenar_y_loggear(
    GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "GradientBoosting_lr01",
    {"model_type": "GradientBoosting", "n_estimators": 100, "learning_rate": 0.1},
    X_train, X_test, y_train, y_test
)

print("Todos los runs loggeados. Revisa la UI: barra izquierda > Experiments")

In [ ]:
# Comparar los runs programaticamente
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.mae ASC"]
)

cols_mostrar = ["tags.mlflow.runName", "metrics.mae", "metrics.rmse", "metrics.r2",
                "params.model_type", "run_id"]
print("Comparacion de runs (ordenados por MAE ascendente):")
display(runs_df[cols_mostrar].rename(columns={"tags.mlflow.runName": "nombre"}))

---
## Seccion 6 — MLflow: registrar el mejor modelo

El **Model Registry** de MLflow permite versionar modelos, asignarles etapas
(Staging, Production) y llevar un historial de deploys.

In [ ]:
# Encontrar el run con menor MAE
mejor_run = runs_df.sort_values("metrics.mae").iloc[0]
mejor_run_id   = mejor_run["run_id"]
mejor_nombre   = mejor_run["tags.mlflow.runName"]
mejor_mae      = round(mejor_run["metrics.mae"], 2)

print(f"Mejor modelo: {mejor_nombre}")
print(f"  MAE   : {mejor_mae}")
print(f"  Run ID: {mejor_run_id}")

In [ ]:
# Registrar el mejor modelo en el Model Registry
MODEL_NAME = "predictor_valor_registros"

model_uri = f"runs:/{mejor_run_id}/model"

result = mlflow.register_model(
    model_uri  = model_uri,
    name       = MODEL_NAME
)

print(f"Modelo registrado en el Model Registry:")
print(f"  Nombre  : {result.name}")
print(f"  Version : {result.version}")
print()
print("Puedes verlo en: barra izquierda > Models")

In [ ]:
# Cargar el modelo desde el registry y hacer una prediccion
modelo_cargado = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/latest")

# Predecir con datos nuevos
categorias_nuevas = pd.DataFrame({"categoria_enc": le.transform(["Ventas", "Tecnologia", "Finanzas"])})
predicciones      = modelo_cargado.predict(categorias_nuevas)

print("Predicciones del modelo registrado:")
for cat, pred in zip(["Ventas", "Tecnologia", "Finanzas"], predicciones):
    print(f"  {cat:15s} -> valor predicho: {pred:.2f}")

---

**Resumen del notebook Databricks Dia 4:**
- Creaste una Delta Table y viste como Delta Lake versiona los datos automaticamente
- Usaste time travel para comparar el estado de la tabla antes y despues de un update
- Ejecutaste un MERGE para hacer upsert sin duplicados
- Loggeaste tres modelos en MLflow y los comparaste por metrica en la UI
- Registraste el mejor modelo en el Model Registry y lo usaste para predecir

Continua con el notebook de Azure ML: `dia4_azureml.ipynb`